In [ ]:
#Importing the necessary libraries
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow.keras import layers

In [ ]:
#Loading the datasets
(train_ds, val_ds, test_ds), metadata = tfds.load(
    'tf_flowers',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
)

In [ ]:
#Showing the number of classes in the dataset
num_classes = metadata.features['label'].num_classes
print(num_classes)

In [ ]:
#Retrieving an image from the data as sample
get_label_name = metadata.features['label'].int2str

image, label = next(iter(train_ds))
_ = plt.imshow(image)
_ = plt.title(get_label_name(label))

In [ ]:
#Resizing and rescaling the images
IMG_SIZE = 180

resize_and_rescale = tf.keras.Sequential([
  layers.Resizing(IMG_SIZE, IMG_SIZE),
  layers.Rescaling(1./255)
])

In [ ]:
#Visualize the result of applying these layers to an image.
result = resize_and_rescale(image)
_ = plt.imshow(result)

In [ ]:
#Verify that the pixels are in the [0, 1] range:
print("Min and max pixel values:", result.numpy().min(), result.numpy().max())

_Data augmenation section_

In [ ]:
#Creating a pre-processing layer and applying them randomly to the same image
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2)
])

In [ ]:
#Add the image to a batch
image = tf.cast(tf.expand_dims(image, 0), tf.float32)

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(9):
  augmented_image = data_augmentation(image)
  ax = plt.subplot(3, 3, i + 1)
  plt.imshow(augmented_image[0])
  plt.axis("off")

In [ ]:
#applying the preprocessing layers to the dataset
batch_size = 32
AUTOTUNE = tf.data.AUTOTUNE

def prepare(ds, shuffle=False, augment=False):
  # Resize and rescale all datasets.
  ds = ds.map(lambda x, y: (resize_and_rescale(x), y), 
              num_parallel_calls=AUTOTUNE)

  if shuffle:
    ds = ds.shuffle(1000)

  # Batch all datasets.
  ds = ds.batch(batch_size)

  # Use data augmentation only on the training set.
  if augment:
    ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), 
                num_parallel_calls=AUTOTUNE)

  # Use buffered prefetching on all datasets.
  return ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
#Preparing the training and testing data with the shuffling and augmentation 
train_ds = prepare(train_ds, shuffle=True, augment=True)
test_ds = prepare(test_ds)

In [ ]:
train_ds = train_ds.map(lambda x, y: (x, tf.one_hot(y, depth=5)))


_All is set to build the CNN model_

In [ ]:
#Using the Sequential Class from Keras to build the model
model = tf.keras.Sequential([
    layers.Input(shape=(180, 180, 3)),
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation = 'softmax')
])

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Fit and train the model with 20 epochs

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs = 20
)